In [ ]:
# Install required packages
import subprocess
import sys

packages = ['scikit-learn', 'tensorflow', 'pandas', 'numpy']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('All packages installed successfully!')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

print('All imports successful!')

In [ ]:
# If test_results_cleaned.csv doesn't exist, generate sample data
import os

if not os.path.exists('test_results_cleaned.csv'):
    print('Creating sample data...')
    np.random.seed(42)
    n_samples = 200
    
    data = {
        'Test ID': [f'TEST_{i:04d}' for i in range(n_samples)],
        'Feature1': np.random.randn(n_samples),
        'Feature2': np.random.randn(n_samples),
        'Feature3': np.random.randn(n_samples),
        'Feature4': np.random.randn(n_samples),
        'Feature5': np.random.randn(n_samples),
        'Suspicious': np.random.randint(0, 2, n_samples)
    }
    
    df = pd.DataFrame(data)
    df.to_csv('test_results_cleaned.csv', index=False)
    print(f'Sample data created: {df.shape[0]} rows, {df.shape[1]} columns')
    print(df.head())
else:
    print('test_results_cleaned.csv already exists!')

In [ ]:
# Load data
_df = pd.read_csv('test_results_cleaned.csv')
print(f'Data shape: {_df.shape}')
print(f'Columns: {_df.columns.tolist()}')

# Drop non-numeric identifier columns
_df = _df.drop(columns=['Test ID'], errors='ignore')

# Split input/target
X = _df.drop(columns=['Suspicious'])
y = _df['Suspicious']

print(f'\nFeatures shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# Stratified cross-validation setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Prepare evaluation loop
fold_scores = []
fold_reports = []

fold_num = 0
for train_index, test_index in skf.split(X, y):
    fold_num += 1
    print(f'\n=== Fold {fold_num} ===')
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Scale the data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Reshape for LSTM: (samples, time_steps, features)
    # Create time steps by stacking features
    n_features = X_train_scaled.shape[1]
    X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], 1, n_features))
    X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], 1, n_features))

    print(f'X_train shape after reshape: {X_train_reshaped.shape}')
    print(f'X_test shape after reshape: {X_test_reshaped.shape}')

    # Build LSTM model
    model = Sequential([
        LSTM(64, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2]), return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    # Early stopping
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0)

    # Train model
    history = model.fit(
        X_train_reshaped,
        y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    # Evaluate
    loss, acc = model.evaluate(X_test_reshaped, y_test, verbose=0)
    fold_scores.append(acc)
    
    # Get predictions for classification report
    y_pred = (model.predict(X_test_reshaped, verbose=0) > 0.5).astype(int).flatten()
    
    print(f'Fold {fold_num} - Loss: {loss:.4f}, Accuracy: {acc:.4f}')
    print(f'Epochs trained: {len(history.history["loss"])}')

print(f'\n' + "="*50)
print(f'Average accuracy across folds: {np.mean(fold_scores):.4f}')
print(f'Std deviation: {np.std(fold_scores):.4f}')
print(f'Min accuracy: {np.min(fold_scores):.4f}')
print(f'Max accuracy: {np.max(fold_scores):.4f}')